# 실습 2. 리뷰 감성 분류 — *불만 리뷰 자동 분류기* 🛎️

> **상황.** 우리 쇼핑몰에는 리뷰가 매일 수천~수만 건씩 쌓입니다.
> 이 중 **불만(부정) 리뷰**를 빨리 못 찾으면 → 환불·악평·고객 이탈로 번집니다.
> 그렇다고 사람이 매일 전부 읽을 수는 없습니다.
>
> **오늘 할 일.** 리뷰 글을 보고 **긍정/부정을 자동으로 맞히는 기계(모델)** 를 만들어,
> 불만 리뷰만 빠르게 골라내 **CS·마케팅이 곧바로 대응**하게 합니다.

---

**이 노트북은 코드를 몰라도 됩니다.** 각 셀을 위에서부터 차례로 **▶ 실행**하면서,
코드보다 **결과(출력)** 를 읽어보세요. 직접 문장을 바꿔 넣어볼 수 있는 칸도 있습니다.

## 🧠 아이디어 — 기계는 어떻게 '글의 감정'을 배울까?

우리에겐 과거 리뷰가 잔뜩 있고, 각 리뷰에는 **별점**이 달려 있습니다.
별점은 **"이 글은 긍정/부정"이라는 정답** 역할을 합니다.

- ⭐⭐⭐⭐⭐ / ⭐⭐⭐⭐  →  **긍정**
- ⭐⭐ / ⭐  →  **부정**
- ⭐⭐⭐ (애매한 중립)  →  이번엔 제외

그래서 우리는 **"이런 글에는 이런 별점이 달리더라"** 를 기계에게 잔뜩 보여줍니다.
그러면 기계는 **별점이 없는 새 글이 와도** 글만 보고 긍정/부정을 맞히게 됩니다.

> 비유하자면 — **정답지(별점)가 있는 과거 문제로 공부시켜서, 정답지 없는 새 문제(별점 없는 새 글)를 풀게** 하는 것입니다.
> 고객 문의·SNS 댓글처럼 **별점이 없는 글**이 세상엔 훨씬 많으니까요.

## 1단계. 데이터 살펴보기 — 우리가 가진 리뷰는?

먼저 과거 리뷰가 몇 건이고, 긍정/부정이 어떤 비율인지 눈으로 봅니다.

In [7]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")

# 별점을 '정답'으로: 4~5점=긍정, 1~2점=부정, 3점(중립)은 이번엔 제외
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]
df["감정"] = df["rating"].apply(lambda r: "긍정 🙂" if r >= 4 else "부정 😡")

print(f"전체 리뷰: {len(df):,}건")
print(df["감정"].value_counts())
print("\n📋 실제 리뷰 예시 5개 -------------------")
for _, row in df.sample(5, random_state=1).iterrows():
    print(f"[{row['감정']}]  {row['clean_text'][:40]}")

전체 리뷰: 3,280건
감정
긍정 🙂    1899
부정 😡    1381
Name: count, dtype: int64

📋 실제 리뷰 예시 5개 -------------------
[긍정 🙂]  사이즈도 딱 맞고 마감이 깔끔해요
[긍정 🙂]  색감이 사진이랑 똑같아서 만족합니다
[부정 😡]  한 번 쓰고 고장났습니다 환불 원해요
[긍정 🙂]  포장도 꼼꼼하고 품질이 기대 이상이에요
[부정 😡]  생각보다 너무 작고 부실해요


In [8]:
# 긍정/부정 비율을 간단한 막대로 (별도 설치 없이 바로 보입니다)
분포 = df["감정"].value_counts()
최대 = 분포.max()
for 감정, 개수 in 분포.items():
    막대 = "█" * int(개수 / 최대 * 30)
    print(f"{감정}  {막대} {개수:,}건")

긍정 🙂  ██████████████████████████████ 1,899건
부정 😡  █████████████████████ 1,381건


## 2단계. 기계 학습시키기 ⚙️

이제 위 데이터로 **기계에게 "글 → 감정" 맞히는 법을 가르칩니다.**

> 🔧 **아래 셀은 기계가 공부하는 부분입니다. 내용을 이해할 필요 없이 그냥 ▶ 실행만 하세요.**
> (어떻게 작동하는지는 같은 과정의 개발자용 노트북 `02_sentiment.ipynb`에서 자세히 다룹니다.)

실행하면 **"학습 완료! 정확도 OO%"** 가 나옵니다 — 기계가 과거 리뷰로 공부를 마친 것입니다.

In [9]:
# ⚙️ [그냥 실행하세요] 기계를 학습시키는 부분 -------------------------------
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df["clean_text"].fillna("")
y = (df["rating"] >= 4).astype(int)          # 긍정=1, 부정=0
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = Pipeline([
    ("vec", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000)),
])
model.fit(X_tr, y_tr)                          # ← 기계가 공부하는 순간
정확도 = accuracy_score(y_te, model.predict(X_te))


def 판정(문장: str):
    """리뷰 한 줄을 넣으면 (감정, 확신도%)를 돌려준다."""
    p_긍정 = model.predict_proba([문장])[0][1]   # 긍정일 확률
    if p_긍정 >= 0.5:
        return "긍정 🙂", round(p_긍정 * 100)
    return "부정 😡", round((1 - p_긍정) * 100)


print(f"✅ 학습 완료! 새 리뷰를 맞히는 정확도: {정확도:.0%}")
print("이제 아래에서 직접 리뷰를 넣어 테스트할 수 있습니다 👇")

✅ 학습 완료! 새 리뷰를 맞히는 정확도: 100%
이제 아래에서 직접 리뷰를 넣어 테스트할 수 있습니다 👇


## 3단계. ⭐ 직접 테스트해보기

학습된 기계에게 리뷰 문장을 주면 **긍정/부정과 확신도(%)** 를 답합니다.
먼저 예시 문장들로 확인해볼까요?

In [10]:
예시_리뷰 = [
    "배송도 빠르고 품질이 기대 이상이에요 또 살게요",
    "한 번 쓰고 고장났습니다 최악이에요 환불 원해요",
    "포장 꼼꼼하고 색감도 사진이랑 똑같아 만족합니다",
    "일주일이나 걸렸고 냄새도 심해서 너무 별로",
]

for 리뷰 in 예시_리뷰:
    감정, 확신도 = 판정(리뷰)
    print(f"{감정} (확신 {확신도}%)  ←  {리뷰}")

긍정 🙂 (확신 98%)  ←  배송도 빠르고 품질이 기대 이상이에요 또 살게요
부정 😡 (확신 99%)  ←  한 번 쓰고 고장났습니다 최악이에요 환불 원해요
긍정 🙂 (확신 97%)  ←  포장 꼼꼼하고 색감도 사진이랑 똑같아 만족합니다
부정 😡 (확신 99%)  ←  일주일이나 걸렸고 냄새도 심해서 너무 별로


### ✍️ 이번엔 직접 써보세요

아래 따옴표 안의 문장을 **여러분이 원하는 리뷰로 바꿔** 실행해 보세요.

In [11]:
내_리뷰 = "별로"

감정, 확신도 = 판정(내_리뷰)
print(f"판정 결과 →  {감정}  (확신 {확신도}%)")

판정 결과 →  부정 😡  (확신 90%)


## 4단계. 🛎️ 실전 — 오늘 들어온 리뷰에서 '불만'만 자동으로 골라내기

이게 바로 비즈니스에서 쓰는 모습입니다.
오늘 새로 들어온 리뷰 더미를 기계에 통과시켜, **부정 리뷰만 추려 "CS 확인 목록"** 을 만듭니다.
담당자는 이 목록만 보면 되니, 전부 읽을 필요가 없습니다.

In [12]:
# '오늘 새로 들어온 리뷰'라고 가정 (실제론 그날 수집된 리뷰가 들어옵니다)
오늘_리뷰 = df["clean_text"].sample(12, random_state=7).tolist()

결과 = []
for 리뷰 in 오늘_리뷰:
    감정, 확신도 = 판정(리뷰)
    결과.append({"감정": 감정, "확신도(%)": 확신도, "리뷰": 리뷰[:45]})

표 = pd.DataFrame(결과)
불만목록 = 표[표["감정"].str.startswith("부정")].sort_values("확신도(%)", ascending=False)

print(f"오늘 리뷰 {len(오늘_리뷰)}건 중 🚨 불만 {len(불만목록)}건 — CS 확인 필요:\n")
불만목록.reset_index(drop=True)

오늘 리뷰 12건 중 🚨 불만 4건 — CS 확인 필요:



,감정,확신도(%),리뷰
0,부정 😡,98,냄새가 너무 심해서 못 쓰겠어요
1,부정 😡,98,생각보다 너무 작고 부실해요
2,부정 😡,98,한 번 쓰고 고장났습니다 환불 원해요
3,부정 😡,98,사진과 색이 완전히 달라서 실망했어요


## 5단계. ⚠️ 기계도 틀립니다 — '반어/비꼼'

기계는 단어를 보고 판단하기 때문에, **칭찬하는 단어로 비꼬는 문장**에 약합니다.
다음 문장들을 보면, 사람은 부정인 걸 바로 알지만 기계는 헷갈려 합니다.

In [13]:
반어_리뷰 = [
    "와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

for 리뷰 in 반어_리뷰:
    감정, 확신도 = 판정(리뷰)
    print(f"{감정} (확신 {확신도}%)  ←  {리뷰}")

print("\n👉 사람에겐 명백한 '부정'인데 기계는 '긍정'이라 착각하기 쉽습니다.")
print("   이런 미묘한 표현을 잡으려면 더 똑똑한 도구(LLM)가 필요합니다 → 실습 3에서 비교합니다.")

긍정 🙂 (확신 78%)  ←  와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다
긍정 🙂 (확신 56%)  ←  품질 최고예요~ 환불하고 싶을 만큼
부정 😡 (확신 90%)  ←  빠른 배송 감사합니다 일주일이나 걸려서요

👉 사람에겐 명백한 '부정'인데 기계는 '긍정'이라 착각하기 쉽습니다.
   이런 미묘한 표현을 잡으려면 더 똑똑한 도구(LLM)가 필요합니다 → 실습 3에서 비교합니다.


## 📌 오늘의 정리 — 비즈니스 관점

**우리가 만든 것:** 리뷰 글만 보고 긍정/부정을 맞혀, **불만 리뷰를 자동으로 추려내는** 분류기.
→ CS는 불만에 빠르게 대응하고, 마케팅은 긍정 리뷰를 홍보 소재로, 부정은 개선 신호로 씁니다.

**기억할 3가지:**

| | 핵심 메시지 |
|---|---|
| 💰 **싸고 빠르다** | 이 방식은 수만 건을 몇 초·거의 0원에 분류합니다. 사람이 다 읽는 것과 비교해 보세요. |
| 🎯 **"AI 쓰자"가 늘 정답은 아니다** | 단순·대량 분류는 이런 가벼운 도구로 충분합니다. 비싼 AI는 꼭 필요한 곳에만. |
| ⚠️ **기계는 어디선가 틀린다** | 특히 반어/비꼼. 어디서 틀리는지 알고 보완책을 두는 게 진짜 실력입니다. |

> **다음 (실습 3).** 똑같은 분류를 **ChatGPT(LLM)** 로 해보고, *정확도·비용·속도·미묘한 표현 이해력* 을 직접 비교합니다.
> "언제 가벼운 모델, 언제 비싼 AI를 써야 돈이 안 새는가" 를 판단하는 눈을 기르는 게 목표입니다.

In [14]:
# `model` 변수가 메모리에 없으면 ../models/sentiment_model.pkl에서 불러옵니다
from pathlib import Path
import pickle

models_path = Path('..') / 'models' / 'sentiment_model.pkl'
if 'model' in globals():
    print('`model` is already in memory.')
else:
    if models_path.exists():
        with open(models_path, 'rb') as f:
            model = pickle.load(f)
        print(f'Loaded model from {models_path}')
    else:
        raise NameError('`model` not found in memory and no saved model at ../models/sentiment_model.pkl. Run the training cell first.')


`model` is already in memory.


In [15]:
import os
import pickle
from pathlib import Path

# 저장 경로 설정 (노트북에서 상위 폴더 ../models)
models_dir = Path('..') / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / 'sentiment_model.pkl'

# `model` 변수가 학습 완료되어 존재하는지 확인
if 'model' not in globals():
    raise NameError('`model` 변수가 존재하지 않습니다. 학습 셀을 먼저 실행하세요.')

# 안전하게 저장: pickle 시도, 실패 시 joblib로 대체
save_error = None
try:
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'Model saved to: {model_path}')
except Exception as e:
    save_error = e
    print('pickle 저장 실패, joblib로 시도합니다:', e)
    try:
        from joblib import dump
        dump(model, str(model_path))
        print(f'Model saved to: {model_path} (joblib)')
        save_error = None
    except Exception as e2:
        print('joblib 저장도 실패했습니다:', e2)
        # 최종 오류 출력
        raise RuntimeError(f'모델 저장 실패: pickle error={save_error}, joblib error={e2}')


Model saved to: ../models/sentiment_model.pkl


In [16]:
# 바로 실행 가능한 추론 셀: 저장된 모델을 찾아 샘플 문장으로 예측합니다
from pathlib import Path
import pickle

# 모델 파일 후보 경로 (노트북/레포 루트 모두 커버)
candidates = [Path('models/sentiment_model.pkl'), Path('..') / 'models' / 'sentiment_model.pkl']
model_path = next((p for p in candidates if p.exists()), None)
if model_path is None:
    raise FileNotFoundError(f'No saved model found. Checked: {candidates}')

with open(model_path, 'rb') as f:
    model = pickle.load(f)
print('Loaded model from', model_path, 'type=', type(model))

# 테스트용 샘플 문장들 (원하시면 수정하세요)
samples = [
    '배송이 빨라서 만족합니다',
    '상품이 도착했는데 고장났어요 환불해주세요',
    '포장도 꼼꼼하고 색감도 좋아요',
]

for s in samples:
    try:
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba([s])[0][1]
            label = '긍정 🙂' if prob >= 0.5 else '부정 😡'
            print(f"{s} -> {label} (긍정확률 {prob:.3f})")
        else:
            pred = model.predict([s])[0]
            print(f"{s} -> pred: {pred}")
    except Exception as e:
        print('Inference error for:', s, '->', e)


Loaded model from ../models/sentiment_model.pkl type= <class 'sklearn.pipeline.Pipeline'>
배송이 빨라서 만족합니다 -> 긍정 🙂 (긍정확률 0.929)
상품이 도착했는데 고장났어요 환불해주세요 -> 긍정 🙂 (긍정확률 0.563)
포장도 꼼꼼하고 색감도 좋아요 -> 긍정 🙂 (긍정확률 0.939)
